# Diabetes Prediction

Notebook de prediccion de diabetes basado en `04_final_diabetes_random_forest.ipynb`, usando el archivo `diabetes.csv` ubicado en `Base-de-datos`.

El flujo incluye exploracion rapida, limpieza de valores invalidos, feature engineering, entrenamiento de Random Forest optimizado y evaluacion del modelo.


## 1. Importar librerías

Se importan las librerías necesarias para análisis de datos, visualización, preparación del dataset, entrenamiento y evaluación del modelo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay
)

sns.set_style("whitegrid")
%matplotlib inline

## 2. Funciones de limpieza y feature engineering

Estas funciones replican la lógica usada en el notebook de limpieza:

- Reemplazar valores `0` inválidos por `NaN`.
- Imputar valores faltantes con la mediana.
- Crear variables categóricas útiles para el modelo, como rangos de BMI, edad, glucosa, insulina y presión arterial.

Se incluyen directamente en este notebook para que sea más fácil ejecutarlo sin depender de archivos externos como `utils.py`.

In [ ]:
def replace_zeros_with_nan(df, columns):
    """Reemplaza valores 0 por NaN en columnas donde 0 no es un valor válido."""
    df_copy = df.copy()
    for col in columns:
        df_copy[col] = df_copy[col].replace(0, np.nan)
    return df_copy


def impute_missing(df, strategy="median"):
    """Imputa valores faltantes usando mediana o media."""
    df_copy = df.copy()
    numeric_cols = df_copy.select_dtypes(include=[np.number]).columns

    for col in numeric_cols:
        if df_copy[col].isnull().sum() > 0:
            if strategy == "median":
                df_copy[col] = df_copy[col].fillna(df_copy[col].median())
            elif strategy == "mean":
                df_copy[col] = df_copy[col].fillna(df_copy[col].mean())
            else:
                raise ValueError("strategy debe ser 'median' o 'mean'")

    return df_copy


def create_features(df):
    """Crea nuevas variables categóricas a partir de variables numéricas clínicas."""
    df_copy = df.copy()

    df_copy["BMI_Category"] = pd.cut(
        df_copy["BMI"],
        bins=[0, 18.5, 25, 30, 100],
        labels=["Underweight", "Normal", "Overweight", "Obese"]
    )

    df_copy["AgeGroup"] = pd.cut(
        df_copy["Age"],
        bins=[0, 30, 45, 60, 100],
        labels=["Young", "Adult", "Middle-aged", "Senior"]
    )

    df_copy["Glucose_Category"] = pd.cut(
        df_copy["Glucose"],
        bins=[0, 100, 125, 200],
        labels=["Normal", "Prediabetes", "Diabetes"]
    )

    df_copy["Insulin_Category"] = pd.cut(
        df_copy["Insulin"],
        bins=[0, 16, 166, 1000],
        labels=["Low", "Normal", "High"]
    )

    df_copy["BP_Category"] = pd.cut(
        df_copy["BloodPressure"],
        bins=[0, 80, 90, 200],
        labels=["Normal", "Elevated", "High"]
    )

    return df_copy

## 3. Cargar el dataset

Se carga el dataset original de diabetes. Si tu archivo está en otra ubicación, solo cambia la ruta en la variable `DATA_PATH`.

In [ ]:
DATA_PATH_CANDIDATES = [
    Path("../Base-de-datos/diabetes.csv"),
    Path("Base-de-datos/diabetes.csv"),
]

DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No se encontro diabetes.csv en Base-de-datos. Revisa la ruta del archivo.")

df = pd.read_csv(DATA_PATH)

print(f"Dataset cargado desde: {DATA_PATH}")
print(f"Filas y columnas originales: {df.shape}")
df.head()


## 4. Exploración rápida de datos

Aunque el notebook final no repite todo el EDA, sí conviene revisar información básica:

- Tipos de datos.
- Valores faltantes.
- Distribución de la variable objetivo `Outcome`.
- Correlación inicial entre variables.

Esto ayuda a confirmar que los datos se cargaron correctamente antes de limpiar y entrenar.

In [ ]:
df.info()

In [ ]:
print("Valores faltantes por columna:")
display(df.isnull().sum())

print("\nDistribución de Outcome:")
display(df["Outcome"].value_counts())

plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="Outcome")
plt.title("Distribución de la variable objetivo")
plt.xlabel("Outcome: 0 = No diabetes, 1 = Diabetes")
plt.ylabel("Cantidad de pacientes")
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación inicial")
plt.show()

## 5. Limpieza de datos

En este dataset hay columnas clínicas donde el valor `0` no tiene sentido, por ejemplo:

- `Glucose`
- `BloodPressure`
- `SkinThickness`
- `Insulin`
- `BMI`

Por eso se reemplazan esos ceros por `NaN` y después se imputan usando la **mediana**, ya que es más resistente a valores extremos que la media.

In [ ]:
zero_invalid_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Cantidad de ceros antes de la limpieza:")
for col in zero_invalid_cols:
    print(f"{col:20s}: {(df[col] == 0).sum()}")

In [ ]:
df_clean = replace_zeros_with_nan(df, zero_invalid_cols)

print("Valores faltantes después de reemplazar ceros por NaN:")
display(df_clean.isnull().sum())

df_clean = impute_missing(df_clean, strategy="median")

print("\nValores faltantes después de imputar:")
display(df_clean.isnull().sum())

## 6. Feature Engineering

Se crean nuevas variables categóricas a partir de columnas numéricas. Esto permite que el modelo aproveche patrones más interpretables, como rangos de BMI, edad o glucosa.

Después, las variables categóricas se convierten a variables numéricas usando `pd.get_dummies`, porque Random Forest necesita trabajar con valores numéricos.

In [ ]:
df_feat = create_features(df_clean)

categorical_cols = [
    "BMI_Category",
    "AgeGroup",
    "Glucose_Category",
    "Insulin_Category",
    "BP_Category"
]

df_encoded = pd.get_dummies(df_feat, columns=categorical_cols, drop_first=True)

print(f"Shape después de feature engineering y encoding: {df_encoded.shape}")
df_encoded.head()

## 7. Separar variables predictoras y variable objetivo

La variable objetivo es `Outcome`.

- `X`: variables usadas para predecir.
- `y`: resultado real que el modelo debe aprender a predecir.

Se usa una separación 80/20 para entrenamiento y prueba. También se usa `stratify=y` para mantener proporciones similares de pacientes con y sin diabetes en ambos conjuntos.

In [ ]:
X = df_encoded.drop("Outcome", axis=1)
y = df_encoded["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test : {y_test.shape}")

## 8. Entrenar solo el mejor modelo: Random Forest optimizado

Según las conclusiones del notebook `03_model_building`, los modelos tipo ensemble tuvieron mejor rendimiento, especialmente **Random Forest** después de ajustar hiperparámetros.

Por eso aquí no se vuelven a entrenar todos los modelos. Solo se entrena el modelo final usando `GridSearchCV`, que prueba varias combinaciones de hiperparámetros y selecciona la mejor con base en el **F1-score**.

Se usa F1-score porque combina precisión y recall, lo cual es útil en problemas médicos donde no solo importa acertar, sino también reducir falsos negativos.

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_

print("Mejores hiperparámetros encontrados:")
print(grid_search.best_params_)
print(f"Mejor F1 promedio en validación cruzada: {grid_search.best_score_:.4f}")

## 9. Evaluación del modelo final

Ahora se evalúa el modelo con datos que no vio durante el entrenamiento (`X_test`). Esto permite medir qué tan bien generaliza a nuevos pacientes.

Las métricas principales son:

- **Accuracy**: porcentaje total de predicciones correctas.
- **Precision**: de los pacientes predichos como diabéticos, cuántos realmente lo eran.
- **Recall**: de los pacientes diabéticos reales, cuántos logró detectar.
- **F1-score**: balance entre precision y recall.
- **ROC-AUC**: capacidad general del modelo para separar ambas clases.

In [ ]:
y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba)
}

metrics_df = pd.DataFrame(metrics, index=["Random Forest Tuned"]).T
metrics_df

In [ ]:
print("Reporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=["No Diabetes", "Diabetes"]))

## 10. Matriz de confusión

La matriz de confusión permite ver cuántos casos fueron clasificados correctamente y cuántos fueron errores.

En este problema, es importante observar especialmente los **falsos negativos**, porque representan pacientes que sí tienen diabetes pero el modelo predijo como no diabéticos.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Diabetes", "Diabetes"]
)

disp.plot(cmap="Blues")
plt.title("Matriz de confusión — Random Forest optimizado")
plt.show()

## 11. Curva ROC

La curva ROC muestra qué tan bien el modelo distingue entre pacientes con diabetes y sin diabetes usando distintas probabilidades de corte.

Mientras más cerca esté la curva de la esquina superior izquierda, mejor es el desempeño del modelo.

In [ ]:
RocCurveDisplay.from_estimator(best_rf, X_test, y_test)
plt.title("Curva ROC — Random Forest optimizado")
plt.show()

## 12. Importancia de variables

Random Forest permite revisar qué variables fueron más importantes para tomar decisiones. Esto no significa causalidad, pero ayuda a interpretar qué características tuvieron mayor peso en las predicciones.

De acuerdo con el análisis previo, variables relacionadas con **glucosa**, **BMI** y **edad** suelen ser de las más relevantes para predecir diabetes.

In [ ]:
feature_importance = pd.Series(
    best_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
feature_importance.tail(15).plot(kind="barh")
plt.title("Top 15 variables más importantes — Random Forest optimizado")
plt.xlabel("Importancia")
plt.show()

## 13. Predicción de ejemplo

Finalmente, se toma un paciente del conjunto de prueba y se genera una predicción individual.

El modelo devuelve:

- La clase predicha.
- La probabilidad de no diabetes.
- La probabilidad de diabetes.

In [ ]:
sample = X_test.iloc[[0]]

prediction = best_rf.predict(sample)[0]
probability = best_rf.predict_proba(sample)[0]

print("Predicción:", "Diabetes" if prediction == 1 else "No diabetes")
print(f"P(No diabetes): {probability[0]:.3f}")
print(f"P(Diabetes)   : {probability[1]:.3f}")

## 14. Conclusión

En este notebook se integró el proceso completo para construir un modelo de predicción de diabetes:

1. Se revisaron los datos de forma general.
2. Se limpiaron valores inválidos en variables clínicas.
3. Se imputaron valores faltantes usando la mediana.
4. Se crearon nuevas variables mediante feature engineering.
5. Se entrenó únicamente el mejor modelo identificado previamente: **Random Forest optimizado**.
6. Se evaluó el modelo con métricas de clasificación, matriz de confusión, curva ROC e importancia de variables.

La razón principal para usar Random Forest es que fue el modelo seleccionado en el notebook de construcción de modelos, ya que tuvo buen desempeño y mejoró después del ajuste de hiperparámetros con `GridSearchCV`.